# FLUT — Massachusetts sportsbook signal (FanDuel)

**Decision question:** Does Massachusetts regulatory evidence indicate that FanDuel’s 2026 sportsbook trajectory is strengthening or weakening relative to the market and 2025?

Narrow evidence check for further FLUT investigation. Not a nationwide proxy, valuation model, or trading recommendation.


Read this notebook in three passes: approval/coverage and reconciliation, monthly and pooled comparisons, then accounting decompositions. All money is USD; shares and hold are ratios of pooled dollars, not average monthly percentages.

Market Accrual Win growth and FanDuel share changes are separate observations. FanDuel revenue = handle × hold; revenue share = handle share × relative hold. These identities explain arithmetic, not customer migration or causes. Source references and missing/duplicate checks precede interpretation. The scoped approval below is historical and is not renewed by these notebook edits.


## 0. Scope, identity, and scoped analyst approval

| Field | Value |
|---|---|
| State | Massachusetts (`MA`) |
| Product | `online_sports_betting` |
| Channel | `online` |
| Frequency | `monthly` |
| Metric | `gross_revenue` (native label: **Accrual Win**) |
| Window | January–July 2026 vs January–July 2025 (14 retained MGC months) |
| Operator | FanDuel via exact name `FanDuel` (not Fanatics) |
| Database | `data/staging/gaming_nationwide.sqlite` (read-only) |

**Commit binding**
- `approved_pipeline_commit` = `2e752d47e810b7aaf583d534ab5a411aa7fa1c1f` — the data-pipeline baseline whose source/config files must still match.
- `current_analysis_commit` = current `HEAD` — may differ for notebook/documentation-only work.
- Staging SHA-256 must remain `023ca5e8e4c16ff0981a2783dabcedf9399939e0241701b0394bc0277eff6ce9`.
- The 14 retained MGC source SHA-256 values must match the approved map.

The notebook fails closed if staging, period hashes, scope fields, or pipeline source/config files diverge from the approved baseline. It does **not** require `current_analysis_commit == approved_pipeline_commit`.


In [ ]:
from __future__ import annotations

import hashlib
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if (ROOT / "gaming" / "src").is_dir():
    ROOT = ROOT / "gaming"
if not (ROOT / "src").exists():
    ROOT = ROOT.parent

import sys
sys.path.insert(0, str(ROOT / "src"))

from variant_gaming.storage import connect_readonly
from variant_gaming.states.massachusetts import RECONCILE_TOLERANCE

DATABASE_FILE = "data/staging/gaming_nationwide.sqlite"
ORIGINAL_DB = ROOT / "data" / "gaming.sqlite"
STAGING_DB = ROOT / DATABASE_FILE
STATE = "MA"
VERTICAL = "online_sports_betting"
CHANNEL = "online"
FREQUENCY = "monthly"
METRIC = "gross_revenue"  # Accrual Win
FANDUEL_OPERATOR = "FanDuel"  # exact reviewed mapping; excludes Fanatics
PERIODS_CURRENT = pd.period_range("2026-01", "2026-07", freq="M")
PERIODS_PRIOR = pd.period_range("2025-01", "2025-07", freq="M")
REQUIRED_PERIODS = [p.to_timestamp() for p in list(PERIODS_PRIOR) + list(PERIODS_CURRENT)]
CENT = 0.01

# Colorblind-readable Okabe–Ito (do not rely on red/green alone).
C_BLUE = "#0072B2"
C_ORANGE = "#E69F00"
C_SKY = "#56B4E9"
C_PURPLE = "#CC79A7"
C_GRAY = "#4D4D4D"
C_BLACK = "#000000"

EXPECTED_STAGING_SHA256 = "023ca5e8e4c16ff0981a2783dabcedf9399939e0241701b0394bc0277eff6ce9"
APPROVED_PIPELINE_COMMIT = "2e752d47e810b7aaf583d534ab5a411aa7fa1c1f"
# Pipeline source/config files that must still match the approved baseline.
# Notebook/documentation-only commits may move HEAD without failing this gate.
PIPELINE_BIND_PATHS = (
    "src/variant_gaming/consolidate.py",
    "src/variant_gaming/storage.py",
    "src/variant_gaming/states/massachusetts.py",
    "config/state_metric_notes.csv",
    "config/state_gaming_source_inventory.csv",
)
APPROVED_PERIOD_SHA256 = {
    pd.Timestamp("2025-01-01"): "dbf48a7935d3664a8aa23cb052b4b3743e9b72ac11da60f2038b0e4a3ed9e196",
    pd.Timestamp("2025-02-01"): "f0c820052f040ce5edbcf830f238145e80263262e2643b68e21fdb05380383a5",
    pd.Timestamp("2025-03-01"): "bf9c0bc5f97e03620b352b073d43e0c88f90286bb8fecc95a62ce1f868e11d3a",
    pd.Timestamp("2025-04-01"): "6254c3063b5fd240c56a4a3b471333f6126a1f548d598a60fc5bcdd5064cdada",
    pd.Timestamp("2025-05-01"): "c171671d461b93516ff2a50341cd221ffe692c4ddf8c58c628ceb2f95f3a4abd",
    pd.Timestamp("2025-06-01"): "e39e3878cf0af5cbfa9140679dd66f3915d6ed0a6028e05f8c0223795cf3693a",
    pd.Timestamp("2025-07-01"): "8d365d618538cc847b0bda6c45d749917c37bf1a12b95bc7ca17f441dae379fa",
    pd.Timestamp("2026-01-01"): "f79abaa5becc563ed7459162b653355048c6e30ab4f2a3542e878f3f969d66ff",
    pd.Timestamp("2026-02-01"): "9c42d82664073e7de41db9170386ad8a2094c949f4d69bcf8233607324d791ba",
    pd.Timestamp("2026-03-01"): "b363f98a625620f672f81967942b966fa7930244b2474e5a39173b6c7525d74c",
    pd.Timestamp("2026-04-01"): "877b5e94c0d2240561b24dc41579e5f2617cad9f0d9428f8161ef99d92bd1d5c",
    pd.Timestamp("2026-05-01"): "a6f57e1571017921d1a11fa6987fbcd98c774bc556eafcb127bbe9951538c96a",
    pd.Timestamp("2026-06-01"): "6d2e3ac63934f1d6169a8861749b85749250873d98e8c628283e788415ea575a",
    pd.Timestamp("2026-07-01"): "ccac154845a1a2e63dac0a33a9d725edf6eeaa2f2037365c9175596c92891517",
}

# Explicit scoped analyst approval for THIS window only.
# Does not approve other periods, states, channels, or metrics.
# Metric-note definition_applies_from / definition_applies_to remain unknown.
ANALYST_APPROVAL = {
    "approved": True,
    "approver": "human analyst (current task)",
    "scope_state": "MA",
    "scope_vertical": "online_sports_betting",
    "scope_channel": "online",
    "scope_frequency": "monthly",
    "scope_metric": "gross_revenue",
    "scope_metric_label": "Accrual Win",
    "scope_window": "2025-01 through 2026-07 (14 retained MGC monthly reports)",
    "comparison": "January-July 2026 versus January-July 2025",
    "approved_periods": list(APPROVED_PERIOD_SHA256.keys()),
    "approved_source_sha256": dict(APPROVED_PERIOD_SHA256),
    "approved_staging_sha256": EXPECTED_STAGING_SHA256,
    "approved_pipeline_commit": APPROVED_PIPELINE_COMMIT,
    "notes": (
        "Approval covers only MA online sports betting Accrual Win / gross_revenue "
        "for the 14 retained MGC reports already reviewed in this task. "
        "It does not extend definition validity beyond this window."
    ),
}


def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def git_rev_parse(*args: str) -> str:
    return subprocess.check_output(["git", "rev-parse", *args], cwd=ROOT, text=True).strip()


def git_blob_hash(commit: str, rel_path: str) -> str:
    return subprocess.check_output(
        ["git", "rev-parse", f"{commit}:{rel_path}"], cwd=ROOT, text=True
    ).strip()


def working_tree_blob_hash(rel_path: Path) -> str:
    """Git blob SHA for the working-tree file (respects clean/EOL filters)."""
    return subprocess.check_output(
        ["git", "hash-object", str(rel_path)], cwd=ROOT, text=True
    ).strip()


hash_original_before = sha256(ORIGINAL_DB)
hash_staging_before = sha256(STAGING_DB)
approved_pipeline_commit = APPROVED_PIPELINE_COMMIT
current_analysis_commit = git_rev_parse("HEAD")

pipeline_mismatches = []
for rel in PIPELINE_BIND_PATHS:
    path = ROOT / rel
    if not path.is_file():
        pipeline_mismatches.append(f"{rel}: missing in working tree")
        continue
    baseline = git_blob_hash(approved_pipeline_commit, rel)
    current = working_tree_blob_hash(path)
    if current != baseline:
        pipeline_mismatches.append(f"{rel}: working tree differs from {approved_pipeline_commit[:12]}")

binding_ok = True
binding_reasons = []
if hash_staging_before != EXPECTED_STAGING_SHA256:
    binding_ok = False
    binding_reasons.append(f"staging SHA-256 {hash_staging_before} != approved {EXPECTED_STAGING_SHA256}")
if pipeline_mismatches:
    binding_ok = False
    binding_reasons.extend(pipeline_mismatches)
if (STATE, VERTICAL, CHANNEL, FREQUENCY, METRIC) != (
    ANALYST_APPROVAL["scope_state"],
    ANALYST_APPROVAL["scope_vertical"],
    ANALYST_APPROVAL["scope_channel"],
    ANALYST_APPROVAL["scope_frequency"],
    ANALYST_APPROVAL["scope_metric"],
):
    binding_ok = False
    binding_reasons.append("state/vertical/channel/frequency/metric do not match scoped approval")
if len(APPROVED_PERIOD_SHA256) != 14:
    binding_ok = False
    binding_reasons.append(f"approved period count {len(APPROVED_PERIOD_SHA256)} != 14")
if not binding_ok:
    raise SystemExit("FAIL CLOSED: analyst-approval binding mismatch: " + "; ".join(binding_reasons))

print(f"approved_pipeline_commit: {approved_pipeline_commit}")
print(f"current_analysis_commit:  {current_analysis_commit}")
if current_analysis_commit != approved_pipeline_commit:
    print("  (HEAD differs; allowed when only notebook/docs changed and pipeline files still match)")
else:
    print("  (HEAD matches approved pipeline baseline)")
print(f"pipeline_bind_paths_ok: {len(PIPELINE_BIND_PATHS)} files match {approved_pipeline_commit[:12]}")
print(f"staging_sha256: {hash_staging_before}")
print(f"original_sha256: {hash_original_before}")
print(f"FanDuel exact mapping: {FANDUEL_OPERATOR!r}")
print(f"analyst_approval.approved: {ANALYST_APPROVAL['approved']}")
print(
    "analyst_approval.scope: "
    f"{ANALYST_APPROVAL['scope_state']} / {ANALYST_APPROVAL['scope_vertical']} / "
    f"{ANALYST_APPROVAL['scope_channel']} / {ANALYST_APPROVAL['scope_frequency']} / "
    f"{ANALYST_APPROVAL['scope_metric_label']}"
)
print(f"analyst_approval.window: {ANALYST_APPROVAL['comparison']}")
print(f"analyst_approval.periods: {len(APPROVED_PERIOD_SHA256)} hash-bound MGC months")
print(f"Reading read-only: {STAGING_DB}")

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)


## 1. Metric note and load observations


In [ ]:
notes = pd.read_csv(ROOT / "config/state_metric_notes.csv").fillna("unknown")
ma_gross = notes[
    (notes["state_code"] == STATE)
    & (notes["vertical"] == VERTICAL)
    & (notes["storage_column"] == METRIC)
].copy()
assert len(ma_gross) == 1, "Expected exactly one MA gross_revenue metric-note row"
print("MA Accrual Win / gross_revenue metric note:")
display(ma_gross.T)
print(
    "definition_applies_from/to remain unknown; "
    "scoped analyst approval gate covers only this 14-month window."
)

connection = connect_readonly(STAGING_DB)
try:
    results = pd.read_sql_query(
        """
        SELECT *
        FROM gaming_results
        WHERE state_code = ?
          AND vertical = ?
          AND channel = ?
          AND frequency = ?
          AND period_start >= ?
          AND period_start <= ?
        """,
        connection,
        params=(
            STATE,
            VERTICAL,
            CHANNEL,
            FREQUENCY,
            REQUIRED_PERIODS[0].strftime("%Y-%m-%d"),
            REQUIRED_PERIODS[-1].strftime("%Y-%m-%d"),
        ),
    )
finally:
    connection.close()

results["period_start"] = pd.to_datetime(results["period_start"])
results["period_end"] = pd.to_datetime(results["period_end"])
results = results[results["period_start"].isin(REQUIRED_PERIODS)].copy()
print(f"Loaded rows for 14 months: {len(results)}")


## 2. Eligibility gates (fail closed)

Calculations run only if every gate passes, including scoped analyst approval.


In [ ]:
gates = []


def add_gate(name: str, ok: bool, reason: str) -> None:
    gates.append({"gate": name, "status": "PASS" if ok else "FAIL", "reason": reason})


official = results[results["row_type"].eq("official_statewide_total")].copy()
operators = results[results["row_type"].eq("operator")].copy()
fanduel = operators[operators["operator"].eq(FANDUEL_OPERATOR)].copy()

# Gate 0: scoped analyst approval bound to exact periods, hashes, staging, and commit
off_hash_map = {
    pd.Timestamp(ts): str(h)
    for ts, h in official.drop_duplicates("period_start").set_index("period_start")["source_sha256"].items()
}
approved_hash_mismatch = [
    (ts.date(), APPROVED_PERIOD_SHA256[ts], off_hash_map.get(ts))
    for ts in APPROVED_PERIOD_SHA256
    if off_hash_map.get(ts) != APPROVED_PERIOD_SHA256[ts]
]
unexpected_official = sorted(set(off_hash_map) - set(APPROVED_PERIOD_SHA256))
missing_approved = sorted(set(APPROVED_PERIOD_SHA256) - set(off_hash_map))
gate0 = bool(ANALYST_APPROVAL.get("approved"))
gate0 = gate0 and ANALYST_APPROVAL.get("scope_state") == STATE
gate0 = gate0 and ANALYST_APPROVAL.get("scope_vertical") == VERTICAL
gate0 = gate0 and ANALYST_APPROVAL.get("scope_channel") == CHANNEL
gate0 = gate0 and ANALYST_APPROVAL.get("scope_frequency") == FREQUENCY
gate0 = gate0 and ANALYST_APPROVAL.get("scope_metric") == METRIC
gate0 = gate0 and ANALYST_APPROVAL.get("approved_staging_sha256") == hash_staging_before
gate0 = gate0 and ANALYST_APPROVAL.get("approved_pipeline_commit") == approved_pipeline_commit
gate0 = gate0 and not pipeline_mismatches
gate0 = gate0 and len(APPROVED_PERIOD_SHA256) == 14
gate0 = gate0 and not approved_hash_mismatch and not unexpected_official and not missing_approved
add_gate(
    "0. Scoped analyst approval bound to 14 MGC hashes, staging SHA-256, and pipeline source/config baseline",
    gate0,
    "ok" if gate0 else (
        f"hash_mismatch={approved_hash_mismatch[:3]}; "
        f"missing={[t.date() for t in missing_approved]}; "
        f"unexpected={[t.date() for t in unexpected_official]}; "
        f"pipeline={pipeline_mismatches[:3]}"
    ),
)

official_counts = official.groupby("period_start").size()
missing_official = [ts for ts in REQUIRED_PERIODS if ts not in set(official_counts.index)]
multi_official = official_counts[official_counts.ne(1)]
gate1 = (not missing_official) and multi_official.empty
add_gate(
    "1. One official_statewide_total per month",
    gate1,
    "ok" if gate1 else f"missing={[t.date() for t in missing_official]}; multi={multi_official.to_dict()}",
)

bad_status = official[~official["report_status"].eq("reconciled_printed_total")]
gate2 = bad_status.empty
add_gate(
    "2. Official report_status = reconciled_printed_total",
    gate2,
    "ok" if gate2 else f"bad periods={sorted(bad_status['period_start'].dt.date.astype(str).unique())}",
)

hash_mismatch = []
for ts, off in official.groupby("period_start"):
    off_hash = off["source_sha256"].iloc[0]
    ops = operators[operators["period_start"].eq(ts)]
    if ops.empty:
        hash_mismatch.append((ts.date(), "no operators"))
        continue
    other = ops[~ops["source_sha256"].eq(off_hash)]
    if not other.empty:
        hash_mismatch.append((ts.date(), f"{len(other)} operators with different hash"))
gate3 = not hash_mismatch
add_gate("3. Operator rows share official source hash", gate3, "ok" if gate3 else str(hash_mismatch[:5]))

fd_counts = fanduel.groupby("period_start").size()
missing_fd = [ts for ts in REQUIRED_PERIODS if ts not in set(fd_counts.index)]
multi_fd = fd_counts[fd_counts.ne(1)]
gate4 = (not missing_fd) and multi_fd.empty
add_gate(
    "4. Exactly one FanDuel operator row per period",
    gate4,
    "ok" if gate4 else f"missing={[t.date() for t in missing_fd]}; multi={multi_fd.to_dict()}",
)


def present(frame: pd.DataFrame, cols: list[str]) -> bool:
    return (not frame.empty) and frame[cols].notna().all().all()


need_cols = ["handle", "gross_revenue", "taxable_revenue"]
gate5 = present(official, need_cols) and present(fanduel, need_cols)
add_gate(
    "5. Market and FanDuel handle, Accrual Win, taxable present",
    gate5,
    "ok" if gate5 else "missing handle / gross_revenue / taxable_revenue",
)

freq_ok = results["frequency"].dropna().eq(FREQUENCY).all()
chan_ok = results["channel"].dropna().eq(CHANNEL).all()
labels = sorted(results["reported_revenue_name"].dropna().astype(str).unique())
gate6 = freq_ok and chan_ok and len(labels) == 1
add_gate(
    "6. Frequency, channel, source label match across years",
    gate6,
    "ok" if gate6 else f"freq_ok={freq_ok}; chan_ok={chan_ok}; labels={labels}",
)

statuses = sorted(results["report_status"].dropna().astype(str).unique())
gate7 = all(s in {"ok", "reconciled_printed_total"} for s in statuses)
add_gate("7. No source conflicts", gate7, "ok" if gate7 else f"statuses={statuses}")

tol_handle = RECONCILE_TOLERANCE["wagers_settled"]
tol_rev = RECONCILE_TOLERANCE["accrual_win"]
reconcile_fails = []
for ts, off in official.groupby("period_start"):
    ops = operators[operators["period_start"].eq(ts)]
    off_h = round(float(off["handle"].iloc[0]), 2)
    off_r = round(float(off["gross_revenue"].iloc[0]), 2)
    sum_h = round(float(ops["handle"].sum()), 2)
    sum_r = round(float(ops["gross_revenue"].sum()), 2)
    if abs(sum_h - off_h) > tol_handle:
        reconcile_fails.append((ts.date(), "handle", sum_h, off_h))
    if abs(sum_r - off_r) > tol_rev:
        reconcile_fails.append((ts.date(), "gross_revenue", sum_r, off_r))
gate8 = not reconcile_fails
add_gate(
    "8. Operator sums reconcile to printed statewide",
    gate8,
    "ok" if gate8 else f"fails={reconcile_fails[:5]}",
)

current_months = {pd.Timestamp(p.to_timestamp()) for p in PERIODS_CURRENT}
prior_months = {pd.Timestamp(p.to_timestamp()) for p in PERIODS_PRIOR}
paired = []
pair_ok = True
for cur in sorted(current_months):
    prior = cur - pd.DateOffset(years=1)
    if prior not in prior_months:
        pair_ok = False
    if cur not in set(official["period_start"]) or prior not in set(official["period_start"]):
        pair_ok = False
    paired.append((prior, cur))
gate9 = pair_ok and len(paired) == 7
add_gate("9. Exact prior-year month pairs (Jan-Jul)", gate9, "ok" if gate9 else str(paired))

prior_denoms_ok = True
denom_reasons = []
for prior, cur in paired:
    for label, frame in (("market", official), ("FanDuel", fanduel)):
        row = frame[frame["period_start"].eq(prior)]
        if row.empty:
            prior_denoms_ok = False
            denom_reasons.append(f"{label} missing {prior.date()}")
            continue
        for col in ("handle", "gross_revenue"):
            val = float(row[col].iloc[0])
            if not (val > 0):
                prior_denoms_ok = False
                denom_reasons.append(f"{label} {prior.date()} {col}={val}")
gate10 = prior_denoms_ok
add_gate("10. Prior-period denominators positive", gate10, "ok" if gate10 else "; ".join(denom_reasons[:6]))

gate_table = pd.DataFrame(gates)
display(gate_table)
ALL_GATES_PASSED = gate_table["status"].eq("PASS").all()
print("ALL_GATES_PASSED:", ALL_GATES_PASSED)
if not ALL_GATES_PASSED:
    print("STOP: eligibility failed. No analytical estimates will be calculated.")


## 3. Core monthly results and exact YoY

Shown only when every gate passes. Aggregate share/hold use summed numerators and denominators.


In [ ]:
if not ALL_GATES_PASSED:
    raise SystemExit("Eligibility gates failed; analytical section skipped.")

official_i = official.set_index("period_start").sort_index()
fd_i = fanduel.set_index("period_start").sort_index()

monthly = pd.DataFrame({"period_start": REQUIRED_PERIODS})
monthly["period_start"] = pd.to_datetime(monthly["period_start"])
monthly = monthly.set_index("period_start")
monthly["market_handle"] = official_i["handle"]
monthly["market_accrual_win"] = official_i["gross_revenue"]
monthly["market_taxable"] = official_i["taxable_revenue"]
monthly["market_hold"] = monthly["market_accrual_win"] / monthly["market_handle"]
monthly["fanduel_handle"] = fd_i["handle"]
monthly["fanduel_accrual_win"] = fd_i["gross_revenue"]
monthly["fanduel_taxable"] = fd_i["taxable_revenue"]
monthly["fanduel_hold"] = monthly["fanduel_accrual_win"] / monthly["fanduel_handle"]
monthly["fanduel_handle_share"] = monthly["fanduel_handle"] / monthly["market_handle"]
monthly["fanduel_revenue_share"] = monthly["fanduel_accrual_win"] / monthly["market_accrual_win"]
monthly["fanduel_relative_hold"] = monthly["fanduel_hold"] / monthly["market_hold"]
monthly = monthly.reset_index()


def money(x): return f"${x:,.2f}"
def pct(x): return f"{100 * x:.2f}%"
def signed_pct(x): return f"{x:+.2f}%"
def pp(x): return f"{x:+.2f} pp"


print("=== Monthly market ===")
display(monthly[["period_start", "market_handle", "market_accrual_win", "market_hold"]].assign(
    market_handle=lambda d: d["market_handle"].map(money),
    market_accrual_win=lambda d: d["market_accrual_win"].map(money),
    market_hold=lambda d: d["market_hold"].map(pct),
))

print("=== Monthly FanDuel ===")
display(monthly[[
    "period_start", "fanduel_handle", "fanduel_accrual_win", "fanduel_hold",
    "fanduel_handle_share", "fanduel_revenue_share",
]].assign(
    fanduel_handle=lambda d: d["fanduel_handle"].map(money),
    fanduel_accrual_win=lambda d: d["fanduel_accrual_win"].map(money),
    fanduel_hold=lambda d: d["fanduel_hold"].map(pct),
    fanduel_handle_share=lambda d: d["fanduel_handle_share"].map(pct),
    fanduel_revenue_share=lambda d: d["fanduel_revenue_share"].map(pct),
))

yoy_rows = []
for prior, cur in paired:
    p = monthly.loc[monthly["period_start"].eq(prior)].iloc[0]
    c = monthly.loc[monthly["period_start"].eq(cur)].iloc[0]
    yoy_rows.append({
        "prior_period": prior,
        "period_start": cur,
        "market_handle_yoy_pct": 100 * (c["market_handle"] / p["market_handle"] - 1),
        "market_accrual_win_yoy_pct": 100 * (c["market_accrual_win"] / p["market_accrual_win"] - 1),
        "fanduel_handle_yoy_pct": 100 * (c["fanduel_handle"] / p["fanduel_handle"] - 1),
        "fanduel_accrual_win_yoy_pct": 100 * (c["fanduel_accrual_win"] / p["fanduel_accrual_win"] - 1),
        "fanduel_handle_share_pp": 100 * (c["fanduel_handle_share"] - p["fanduel_handle_share"]),
        "fanduel_revenue_share_pp": 100 * (c["fanduel_revenue_share"] - p["fanduel_revenue_share"]),
        "fanduel_hold_pp": 100 * (c["fanduel_hold"] - p["fanduel_hold"]),
        "market_hold_pp": 100 * (c["market_hold"] - p["market_hold"]),
    })
yoy = pd.DataFrame(yoy_rows)
print("=== Exact monthly YoY ===")
display(yoy.assign(
    market_handle_yoy_pct=lambda d: d["market_handle_yoy_pct"].map(signed_pct),
    market_accrual_win_yoy_pct=lambda d: d["market_accrual_win_yoy_pct"].map(signed_pct),
    fanduel_handle_yoy_pct=lambda d: d["fanduel_handle_yoy_pct"].map(signed_pct),
    fanduel_accrual_win_yoy_pct=lambda d: d["fanduel_accrual_win_yoy_pct"].map(signed_pct),
    fanduel_handle_share_pp=lambda d: d["fanduel_handle_share_pp"].map(pp),
    fanduel_revenue_share_pp=lambda d: d["fanduel_revenue_share_pp"].map(pp),
    fanduel_hold_pp=lambda d: d["fanduel_hold_pp"].map(pp),
    market_hold_pp=lambda d: d["market_hold_pp"].map(pp),
))


def agg_block(months, label):
    sub = monthly[monthly["period_start"].isin(months)]
    m_h = float(sub["market_handle"].sum())
    m_r = float(sub["market_accrual_win"].sum())
    m_t = float(sub["market_taxable"].sum())
    f_h = float(sub["fanduel_handle"].sum())
    f_r = float(sub["fanduel_accrual_win"].sum())
    f_t = float(sub["fanduel_taxable"].sum())
    return {
        "period": label,
        "market_handle": m_h,
        "market_accrual_win": m_r,
        "market_taxable": m_t,
        "market_hold": m_r / m_h,
        "fanduel_handle": f_h,
        "fanduel_accrual_win": f_r,
        "fanduel_taxable": f_t,
        "fanduel_hold": f_r / f_h,
        "fanduel_handle_share": f_h / m_h,
        "fanduel_revenue_share": f_r / m_r,
        "fanduel_relative_hold": (f_r / f_h) / (m_r / m_h),
    }


q1_2025 = [pd.Timestamp(f"2025-{m:02d}-01") for m in (1, 2, 3)]
q1_2026 = [pd.Timestamp(f"2026-{m:02d}-01") for m in (1, 2, 3)]
q2_2025 = [pd.Timestamp(f"2025-{m:02d}-01") for m in (4, 5, 6)]
q2_2026 = [pd.Timestamp(f"2026-{m:02d}-01") for m in (4, 5, 6)]
ytd_2025 = [pd.Timestamp(f"2025-{m:02d}-01") for m in range(1, 8)]
ytd_2026 = [pd.Timestamp(f"2026-{m:02d}-01") for m in range(1, 8)]
jul_2025 = [pd.Timestamp("2025-07-01")]
jul_2026 = [pd.Timestamp("2026-07-01")]

agg_prior = {
    "Q1": agg_block(q1_2025, "Q1 2025"),
    "Q2": agg_block(q2_2025, "Q2 2025"),
    "Jan-Jul": agg_block(ytd_2025, "Jan-Jul 2025"),
    "July": agg_block(jul_2025, "July 2025 (partial Q3)"),
}
agg_cur = {
    "Q1": agg_block(q1_2026, "Q1 2026"),
    "Q2": agg_block(q2_2026, "Q2 2026"),
    "Jan-Jul": agg_block(ytd_2026, "Jan-Jul 2026"),
    "July": agg_block(jul_2026, "July 2026 (partial Q3)"),
}

agg_rows = []
for key in ("Q1", "Q2", "Jan-Jul", "July"):
    p, c = agg_prior[key], agg_cur[key]
    agg_rows.append({
        "key": key,
        "comparison": f"{c['period']} vs {p['period']}",
        "market_handle_yoy_pct": 100 * (c["market_handle"] / p["market_handle"] - 1),
        "market_accrual_win_yoy_pct": 100 * (c["market_accrual_win"] / p["market_accrual_win"] - 1),
        "fanduel_handle_yoy_pct": 100 * (c["fanduel_handle"] / p["fanduel_handle"] - 1),
        "fanduel_accrual_win_yoy_pct": 100 * (c["fanduel_accrual_win"] / p["fanduel_accrual_win"] - 1),
        "fanduel_handle_share_pp": 100 * (c["fanduel_handle_share"] - p["fanduel_handle_share"]),
        "fanduel_revenue_share_pp": 100 * (c["fanduel_revenue_share"] - p["fanduel_revenue_share"]),
        "fanduel_hold_pp": 100 * (c["fanduel_hold"] - p["fanduel_hold"]),
        "market_hold_pp": 100 * (c["market_hold"] - p["market_hold"]),
        "note": "partial Q3 only; not annualized" if key == "July" else "",
        **{f"prior_{k}": p[k] for k in p if k != "period"},
        **{f"current_{k}": c[k] for k in c if k != "period"},
    })
agg = pd.DataFrame(agg_rows)
print("=== Aggregated period comparisons ===")
display(agg[[
    "comparison", "market_handle_yoy_pct", "market_accrual_win_yoy_pct",
    "fanduel_handle_yoy_pct", "fanduel_accrual_win_yoy_pct",
    "fanduel_handle_share_pp", "fanduel_revenue_share_pp",
    "fanduel_hold_pp", "market_hold_pp", "note",
]].assign(
    market_handle_yoy_pct=lambda d: d["market_handle_yoy_pct"].map(signed_pct),
    market_accrual_win_yoy_pct=lambda d: d["market_accrual_win_yoy_pct"].map(signed_pct),
    fanduel_handle_yoy_pct=lambda d: d["fanduel_handle_yoy_pct"].map(signed_pct),
    fanduel_accrual_win_yoy_pct=lambda d: d["fanduel_accrual_win_yoy_pct"].map(signed_pct),
    fanduel_handle_share_pp=lambda d: d["fanduel_handle_share_pp"].map(pp),
    fanduel_revenue_share_pp=lambda d: d["fanduel_revenue_share_pp"].map(pp),
    fanduel_hold_pp=lambda d: d["fanduel_hold_pp"].map(pp),
    market_hold_pp=lambda d: d["market_hold_pp"].map(pp),
))

jan_jul = agg.loc[agg["key"].eq("Jan-Jul")].iloc[0]
print("REPRO_CHECK Jan-Jul:")
print(f"  market handle YoY {jan_jul['market_handle_yoy_pct']:+.2f}%")
print(f"  market Accrual Win YoY {jan_jul['market_accrual_win_yoy_pct']:+.2f}%")
print(f"  FanDuel handle YoY {jan_jul['fanduel_handle_yoy_pct']:+.2f}%")
print(f"  FanDuel Accrual Win YoY {jan_jul['fanduel_accrual_win_yoy_pct']:+.2f}%")
print(f"  FanDuel handle share {100*jan_jul['prior_fanduel_handle_share']:.2f}% -> {100*jan_jul['current_fanduel_handle_share']:.2f}%")
print(f"  FanDuel hold {100*jan_jul['prior_fanduel_hold']:.2f}% -> {100*jan_jul['current_fanduel_hold']:.2f}%")
assert abs(jan_jul["market_handle_yoy_pct"] - 2.77) < 0.05
assert abs(jan_jul["market_accrual_win_yoy_pct"] - 5.61) < 0.05
assert abs(jan_jul["fanduel_handle_yoy_pct"] - (-4.97)) < 0.05
assert abs(jan_jul["fanduel_accrual_win_yoy_pct"] - 1.81) < 0.05
assert abs(100 * jan_jul["prior_fanduel_handle_share"] - 27.11) < 0.05
assert abs(100 * jan_jul["current_fanduel_handle_share"] - 25.07) < 0.05
assert abs(100 * jan_jul["prior_fanduel_hold"] - 11.18) < 0.05
assert abs(100 * jan_jul["current_fanduel_hold"] - 11.98) < 0.05
print("REPRO_CHECK: passed")


## 4. Exact symmetric decomposition (Revenue = Handle × Hold)

```
handle_effect = (H1 - H0) * (hold0 + hold1) / 2
hold_effect   = (hold1 - hold0) * (H0 + H1) / 2
```
Components must reconcile to the revenue change within one cent.


In [ ]:
if not ALL_GATES_PASSED:
    raise SystemExit("Eligibility gates failed.")


def symmetric_product_decomp(x0, y0, x1, y1):
    """Decompose change in x*y into x-effect and y-effect (symmetric)."""
    x_effect = (x1 - x0) * (y0 + y1) / 2
    y_effect = (y1 - y0) * (x0 + x1) / 2
    delta = x1 * y1 - x0 * y0
    return x_effect, y_effect, delta


decomp_rows = []
for key in ("Q1", "Q2", "Jan-Jul", "July"):
    row = agg.loc[agg["key"].eq(key)].iloc[0]
    for who, h0k, r0k, hold0k, h1k, r1k, hold1k in (
        ("FanDuel", "prior_fanduel_handle", "prior_fanduel_accrual_win", "prior_fanduel_hold",
         "current_fanduel_handle", "current_fanduel_accrual_win", "current_fanduel_hold"),
        ("Statewide", "prior_market_handle", "prior_market_accrual_win", "prior_market_hold",
         "current_market_handle", "current_market_accrual_win", "current_market_hold"),
    ):
        h0, r0, hold0 = float(row[h0k]), float(row[r0k]), float(row[hold0k])
        h1, r1, hold1 = float(row[h1k]), float(row[r1k]), float(row[hold1k])
        handle_effect, hold_effect, delta = symmetric_product_decomp(h0, hold0, h1, hold1)
        assert abs((handle_effect + hold_effect) - delta) <= CENT, (key, who, handle_effect, hold_effect, delta)
        assert abs(delta - (r1 - r0)) <= CENT, (key, who, delta, r1 - r0)
        entry = {
            "period": key,
            "entity": who,
            "revenue_change_$": delta,
            "handle_effect_$": handle_effect,
            "hold_effect_$": hold_effect,
            "handle_effect_pct_of_prior_revenue": 100 * handle_effect / r0,
            "hold_effect_pct_of_prior_revenue": 100 * hold_effect / r0,
            "revenue_change_pct_of_prior": 100 * delta / r0,
            "reconcile_error_$": (handle_effect + hold_effect) - delta,
            "note": "partial Q3" if key == "July" else "",
        }
        if who == "FanDuel":
            s0 = float(row["prior_fanduel_handle_share"])
            s1 = float(row["current_fanduel_handle_share"])
            rel0 = float(row["prior_fanduel_relative_hold"])
            rel1 = float(row["current_fanduel_relative_hold"])
            rs0 = float(row["prior_fanduel_revenue_share"])
            rs1 = float(row["current_fanduel_revenue_share"])
            share_effect, relhold_effect, rs_delta = symmetric_product_decomp(s0, rel0, s1, rel1)
            assert abs((share_effect + relhold_effect) - rs_delta) <= 1e-12
            assert abs(rs_delta - (rs1 - rs0)) <= 1e-12
            entry["revshare_change_pp"] = 100 * rs_delta
            entry["handle_share_effect_pp"] = 100 * share_effect
            entry["relative_hold_effect_pp"] = 100 * relhold_effect
        decomp_rows.append(entry)

decomp = pd.DataFrame(decomp_rows)
print("=== Revenue = Handle x Hold decomposition ===")
_decomp_view = decomp[[
    "period", "entity", "revenue_change_$", "handle_effect_$", "hold_effect_$",
    "handle_effect_pct_of_prior_revenue", "hold_effect_pct_of_prior_revenue",
    "revenue_change_pct_of_prior", "reconcile_error_$", "note",
]].copy()
_decomp_view["revenue_change_$"] = _decomp_view["revenue_change_$"].map(money)
_decomp_view["handle_effect_$"] = _decomp_view["handle_effect_$"].map(money)
_decomp_view["hold_effect_$"] = _decomp_view["hold_effect_$"].map(money)
_decomp_view["reconcile_error_$"] = _decomp_view["reconcile_error_$"].map(money)
_decomp_view["handle_effect_pct_of_prior_revenue"] = _decomp_view["handle_effect_pct_of_prior_revenue"].map(signed_pct)
_decomp_view["hold_effect_pct_of_prior_revenue"] = _decomp_view["hold_effect_pct_of_prior_revenue"].map(signed_pct)
_decomp_view["revenue_change_pct_of_prior"] = _decomp_view["revenue_change_pct_of_prior"].map(signed_pct)
display(_decomp_view)

print("=== Revenue share = Handle share x Relative hold (FanDuel) ===")
rs = decomp[decomp["entity"].eq("FanDuel")][[
    "period", "revshare_change_pp", "handle_share_effect_pp", "relative_hold_effect_pp", "note"
]].copy()
display(rs.assign(
    revshare_change_pp=lambda d: d["revshare_change_pp"].map(pp),
    handle_share_effect_pp=lambda d: d["handle_share_effect_pp"].map(pp),
    relative_hold_effect_pp=lambda d: d["relative_hold_effect_pp"].map(pp),
))


## 5. Competitor handle-share bridge (Jan–Jul)

Native reported names only. `ESPN Bet` and `theScore Bet` / `theScore Bet*` are shown separately; any corporate continuity is flagged for analyst review, not assumed.


In [ ]:
if not ALL_GATES_PASSED:
    raise SystemExit("Eligibility gates failed.")


def operator_share_frame(months, label):
    sub = operators[operators["period_start"].isin(months)].copy()
    if sub.empty:
        raise SystemExit(f"FAIL CLOSED: no operator rows for {label}")
    missing_vals = sub[sub["handle"].isna() | sub["gross_revenue"].isna()]
    if not missing_vals.empty:
        raise SystemExit(
            f"FAIL CLOSED: missing reported operator values in {label}: "
            f"{missing_vals[['period_start', 'operator']].to_string(index=False)}"
        )
    mkt_handle = float(official[official["period_start"].isin(months)]["handle"].sum())
    g = sub.groupby("operator", as_index=False).agg(handle=("handle", "sum"))
    if g["handle"].isna().any():
        raise SystemExit(f"FAIL CLOSED: operator handle became missing after aggregation in {label}")
    g["share"] = g["handle"] / mkt_handle
    g["period"] = label
    g["roster"] = "reported in roster"
    return g, mkt_handle


prior_ops, prior_mkt = operator_share_frame(ytd_2025, "Jan-Jul 2025")
cur_ops, cur_mkt = operator_share_frame(ytd_2026, "Jan-Jul 2026")
bridge = prior_ops[["operator", "share", "handle"]].merge(
    cur_ops[["operator", "share", "handle"]],
    on="operator",
    how="outer",
    suffixes=("_prior", "_current"),
)
# Do not fillna(0). An operator absent from a verified complete roster is labeled,
# not converted into a fabricated reported zero.
NOT_IN_ROSTER = "not reported in roster"
bridge["prior_status"] = bridge["handle_prior"].notna().map({True: "reported in roster", False: NOT_IN_ROSTER})
bridge["current_status"] = bridge["handle_current"].notna().map({True: "reported in roster", False: NOT_IN_ROSTER})
# Arithmetic share for an absent-from-roster operator is 0 because that year's
# verified roster already accounts for 100% of statewide handle. This is not a
# reported observation.
prior_share_for_change = bridge["share_prior"].where(bridge["handle_prior"].notna(), 0.0)
current_share_for_change = bridge["share_current"].where(bridge["handle_current"].notna(), 0.0)
bridge["share_pp_change"] = 100 * (current_share_for_change - prior_share_for_change)
bridge = bridge.sort_values("share_pp_change")

share_sum_change = float(bridge["share_pp_change"].sum())
assert abs(share_sum_change) < 0.05, share_sum_change

name_flags = []
for a, b in (("ESPN Bet", "theScore Bet"), ("ESPN Bet", "theScore Bet*"), ("theScore Bet", "theScore Bet*")):
    if a in set(bridge["operator"]) and b in set(bridge["operator"]):
        name_flags.append(
            f"{a!r} and {b!r} both appear as native names; do not silently map. "
            "Corporate continuity requires separately retained evidence."
        )


def _disp_money(val, status):
    return NOT_IN_ROSTER if status == NOT_IN_ROSTER else money(val)


def _disp_pct(val, status):
    return NOT_IN_ROSTER if status == NOT_IN_ROSTER else pct(val)


print("=== Competitor handle-share bridge (Jan-Jul) ===")
display(bridge.assign(
    handle_prior=lambda d: [_disp_money(v, s) for v, s in zip(d["handle_prior"], d["prior_status"])],
    handle_current=lambda d: [_disp_money(v, s) for v, s in zip(d["handle_current"], d["current_status"])],
    share_prior=lambda d: [_disp_pct(v, s) for v, s in zip(d["share_prior"], d["prior_status"])],
    share_current=lambda d: [_disp_pct(v, s) for v, s in zip(d["share_current"], d["current_status"])],
    share_pp_change=lambda d: d["share_pp_change"].map(pp),
)[[
    "operator", "prior_status", "handle_prior", "share_prior",
    "current_status", "handle_current", "share_current", "share_pp_change",
]])
print(f"Sum of operator share changes: {share_sum_change:+.4f} pp (expect ~0)")
for flag in name_flags:
    print("NAME_TRANSITION_FLAG:", flag)

fd_loss = float(bridge.loc[bridge["operator"].eq(FANDUEL_OPERATOR), "share_pp_change"].iloc[0])
gainers = bridge[bridge["share_pp_change"] > 0].sort_values("share_pp_change", ascending=False)
print(f"FanDuel handle-share change: {fd_loss:+.2f} pp")
print("Operators that gained handle share:")
display(gainers.assign(
    share_prior=lambda d: [_disp_pct(v, s) for v, s in zip(d["share_prior"], d["prior_status"])],
    share_current=lambda d: [_disp_pct(v, s) for v, s in zip(d["share_current"], d["current_status"])],
    share_pp_change=lambda d: d["share_pp_change"].map(pp),
)[["operator", "prior_status", "share_prior", "current_status", "share_current", "share_pp_change"]])


## 6. Reported deduction gap (Accrual Win − Taxable Gaming Revenue)

“Reported deduction gap” / “promotions and other reported adjustments.” Not pure promotional spend and not an EBITDA expense.


In [ ]:
if not ALL_GATES_PASSED:
    raise SystemExit("Eligibility gates failed.")

ded_rows = []
for key in ("Q1", "Q2", "Jan-Jul", "July"):
    row = agg.loc[agg["key"].eq(key)].iloc[0]
    for year_side, prefix in (("prior", "prior_"), ("current", "current_")):
        for ent, aw_k, tax_k in (
            ("FanDuel", f"{prefix}fanduel_accrual_win", f"{prefix}fanduel_taxable"),
            ("Statewide", f"{prefix}market_accrual_win", f"{prefix}market_taxable"),
        ):
            aw = float(row[aw_k])
            tax = float(row[tax_k])
            gap = aw - tax
            intensity = gap / aw
            ded_rows.append({
                "period": key,
                "year_side": "2025" if year_side == "prior" else "2026",
                "entity": ent,
                "accrual_win": aw,
                "taxable_gaming_revenue": tax,
                "reported_deduction_gap": gap,
                "reported_deduction_intensity": intensity,
                "note": "partial Q3" if key == "July" else "",
            })

ded = pd.DataFrame(ded_rows)
print("=== Reported deduction gap and intensity ===")
display(ded.assign(
    accrual_win=lambda d: d["accrual_win"].map(money),
    taxable_gaming_revenue=lambda d: d["taxable_gaming_revenue"].map(money),
    reported_deduction_gap=lambda d: d["reported_deduction_gap"].map(money),
    reported_deduction_intensity=lambda d: d["reported_deduction_intensity"].map(pct),
))

jj_fd_2025 = ded[(ded.period == "Jan-Jul") & (ded.year_side == "2025") & (ded.entity == "FanDuel")].iloc[0]
jj_fd_2026 = ded[(ded.period == "Jan-Jul") & (ded.year_side == "2026") & (ded.entity == "FanDuel")].iloc[0]
jj_m_2025 = ded[(ded.period == "Jan-Jul") & (ded.year_side == "2025") & (ded.entity == "Statewide")].iloc[0]
jj_m_2026 = ded[(ded.period == "Jan-Jul") & (ded.year_side == "2026") & (ded.entity == "Statewide")].iloc[0]
assert abs(100 * jj_fd_2025["reported_deduction_intensity"] - 2.17) < 0.05
assert abs(100 * jj_fd_2026["reported_deduction_intensity"] - 2.01) < 0.05
assert abs(100 * jj_m_2025["reported_deduction_intensity"] - 2.32) < 0.05
assert abs(100 * jj_m_2026["reported_deduction_intensity"] - 2.23) < 0.05
print("DEDUCTION_REGRESSION_CHECK: passed")
print(
    f"FanDuel Jan-Jul intensity {100*jj_fd_2025['reported_deduction_intensity']:.2f}% -> "
    f"{100*jj_fd_2026['reported_deduction_intensity']:.2f}%"
)
print(
    f"Statewide Jan-Jul intensity {100*jj_m_2025['reported_deduction_intensity']:.2f}% -> "
    f"{100*jj_m_2026['reported_deduction_intensity']:.2f}%"
)


## 7. Within-year trailing three-month smoothing

Uses only approved months in each year. January and February trailing values are unavailable until three months exist.


In [ ]:
if not ALL_GATES_PASSED:
    raise SystemExit("Eligibility gates failed.")

smooth_rows = []
for year, months in ((2025, ytd_2025), (2026, ytd_2026)):
    sub = monthly[monthly["period_start"].isin(months)].sort_values("period_start").copy()
    for i, row in sub.reset_index(drop=True).iterrows():
        if i < 2:
            continue
        window = sub.iloc[i - 2 : i + 1]
        m_h = float(window["market_handle"].sum())
        m_r = float(window["market_accrual_win"].sum())
        f_h = float(window["fanduel_handle"].sum())
        f_r = float(window["fanduel_accrual_win"].sum())
        smooth_rows.append({
            "year": year,
            "as_of": row["period_start"],
            "fanduel_handle_share_t3m": f_h / m_h,
            "fanduel_revenue_share_t3m": f_r / m_r,
            "fanduel_hold_t3m": f_r / f_h,
            "market_hold_t3m": m_r / m_h,
        })

smooth = pd.DataFrame(smooth_rows)
print("=== Trailing 3-month (within-year, from March) ===")
display(smooth.assign(
    fanduel_handle_share_t3m=lambda d: d["fanduel_handle_share_t3m"].map(pct),
    fanduel_revenue_share_t3m=lambda d: d["fanduel_revenue_share_t3m"].map(pct),
    fanduel_hold_t3m=lambda d: d["fanduel_hold_t3m"].map(pct),
    market_hold_t3m=lambda d: d["market_hold_t3m"].map(pct),
))


## 8. Visual decision summary (six figures)

Detailed tables above remain the audit record. Charts use calculated values only. Massachusetts is not a nationwide FLUT proxy.


In [ ]:
if not ALL_GATES_PASSED:
    raise SystemExit("Eligibility gates failed.")

q1_pp = float(agg.loc[agg["key"].eq("Q1"), "fanduel_handle_share_pp"].iloc[0])
q2_pp = float(agg.loc[agg["key"].eq("Q2"), "fanduel_handle_share_pp"].iloc[0])
july_pp = float(agg.loc[agg["key"].eq("July"), "fanduel_handle_share_pp"].iloc[0])
ytd_pp = float(agg.loc[agg["key"].eq("Jan-Jul"), "fanduel_handle_share_pp"].iloc[0])
fd_jj = decomp[(decomp["period"].eq("Jan-Jul")) & (decomp["entity"].eq("FanDuel"))].iloc[0]
prior_fd_rev = float(jan_jul["prior_fanduel_accrual_win"])
handle_fx = float(fd_jj["handle_effect_$"])
hold_fx = float(fd_jj["hold_effect_$"])
current_fd_rev = float(jan_jul["current_fanduel_accrual_win"])
assert abs((prior_fd_rev + handle_fx + hold_fx) - current_fd_rev) <= CENT

# 1. Level versus direction
fig, ax = plt.subplots(figsize=(7.5, 4.2))
labels_1 = ["Q1", "Q2", "July\n(1 month)", "Jan–Jul\nYTD"]
vals_1 = [q1_pp, q2_pp, july_pp, ytd_pp]
colors_1 = [C_BLUE if v < 0 else C_ORANGE for v in vals_1]
bars = ax.bar(labels_1, vals_1, color=colors_1, width=0.65, zorder=2)
ax.axhline(0, color=C_BLACK, linewidth=0.9, zorder=3)
ax.set_title("Did FanDuel’s MA handle-share gap improve after a weak Q1?")
ax.set_ylabel("Handle-share change vs prior-year period (pp)")
ax.set_xlabel("Comparison window (YoY)")
for bar, v in zip(bars, vals_1):
    offset = 0.12 if v >= 0 else -0.22
    ax.text(bar.get_x() + bar.get_width() / 2, v + offset, f"{v:+.2f} pp", ha="center", va="bottom" if v >= 0 else "top", fontsize=9)
ax.set_ylim(min(vals_1) - 1.0, 2.0)
ax.annotate(
    "July is one month,\nnot a confirmed reversal",
    xy=(2, july_pp),
    xytext=(0.85, 1.2),
    fontsize=8,
    color=C_GRAY,
    ha="center",
    va="bottom",
    arrowprops={"arrowstyle": "->", "color": C_GRAY, "connectionstyle": "arc3,rad=0.2"},
)
fig.tight_layout()
plt.show()
print("YTD position remains cautionary (−2.04 pp). Sequential direction improved Q1→Q2→July; July durability is unproven.")

# 2. Relative handle-growth gap
yoy_plot = yoy.copy()
yoy_plot["month"] = pd.to_datetime(yoy_plot["period_start"]).dt.strftime("%b")
yoy_plot["rel_handle_gap"] = yoy_plot["fanduel_handle_yoy_pct"] - yoy_plot["market_handle_yoy_pct"]
fig, ax = plt.subplots(figsize=(7.5, 4.0))
ax.axhline(0, color=C_BLACK, linewidth=0.9)
ax.plot(yoy_plot["month"], yoy_plot["rel_handle_gap"], marker="o", color=C_BLUE, linewidth=2)
for _, r in yoy_plot.iterrows():
    ax.text(r["month"], r["rel_handle_gap"] + (0.35 if r["rel_handle_gap"] >= 0 else -0.55),
            f"{r['rel_handle_gap']:+.1f}", ha="center", fontsize=8)
ax.set_title("Did FanDuel handle grow faster or slower than the MA market?")
ax.set_ylabel("FanDuel handle YoY minus statewide handle YoY (pp)")
ax.set_xlabel("Matched month (2026 vs 2025)")
fig.tight_layout()
plt.show()
print("Below the zero line, FanDuel undergrew the Massachusetts market. That gap narrowed into July but is still one observation.")

# 3. Matched-month share comparison — two years as separate series (no 2025-07→2026-01 line)
months_abbr = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul"]
share_2025 = monthly[monthly["period_start"].isin(ytd_2025)].sort_values("period_start")
share_2026 = monthly[monthly["period_start"].isin(ytd_2026)].sort_values("period_start")
fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.0), sharey=False)
for ax, col, title, ylab in (
    (axes[0], "fanduel_handle_share", "A. Handle share", "FanDuel handle share (%)"),
    (axes[1], "fanduel_revenue_share", "B. Accrual Win share", "FanDuel Accrual Win share (%)"),
):
    y25 = 100 * share_2025[col].to_numpy()
    y26 = 100 * share_2026[col].to_numpy()
    ax.plot(months_abbr, y25, marker="o", color=C_GRAY, linewidth=2, label="2025")
    ax.plot(months_abbr, y26, marker="o", color=C_BLUE, linewidth=2, label="2026")
    ax.set_title(title)
    ax.set_xlabel("Month (Jan–Jul, matched)")
    ax.set_ylabel(ylab)
    ax.text(0, y25[0], "2025", color=C_GRAY, fontsize=8, va="bottom")
    ax.text(0, y26[0], "2026", color=C_BLUE, fontsize=8, va="top")
fig.suptitle("Was FanDuel’s 2026 MA share below the same month in 2025?", y=1.02)
fig.tight_layout()
plt.show()
print("Each year is its own Jan–Jul series. There is no line connecting July 2025 to January 2026.")

# 4. Revenue decomposition waterfall (calculated, reconcile to the cent)
def money_m(x: float) -> str:
    return f"${x / 1e6:,.2f}m"

fig, ax = plt.subplots(figsize=(8.0, 4.4))
wf_labels = ["2025 Jan–Jul\nAccrual Win", "Handle\neffect", "Hold\neffect", "2026 Jan–Jul\nAccrual Win"]
wf_vals = [prior_fd_rev, handle_fx, hold_fx, current_fd_rev]
starts = [0.0, prior_fd_rev, prior_fd_rev + handle_fx, 0.0]
wf_colors = [C_GRAY, C_BLUE, C_ORANGE, C_PURPLE]
for i, (lab, val, start, col) in enumerate(zip(wf_labels, wf_vals, starts, wf_colors)):
    if i in (0, 3):
        ax.bar(i, val, color=col, width=0.62, zorder=2)
        ax.text(i, val + 4e5, money_m(val), ha="center", fontsize=8)
    else:
        ax.bar(i, val, bottom=start, color=col, width=0.62, zorder=2)
        ax.text(i, start + val + (4e5 if val >= 0 else -8e5), money_m(val), ha="center", fontsize=8)
    if i < 3:
        ax.plot([i + 0.31, i + 0.69], [start + (val if i else val), starts[i + 1] if i < 2 else current_fd_rev],
                color=C_GRAY, linewidth=0.8)
ax.set_xticks(range(4), wf_labels)
ax.set_ylabel("FanDuel Accrual Win (USD)")
ax.set_xlabel("Jan–Jul Massachusetts online sports betting")
ax.set_title("Higher hold offset weaker handle.")
ax.axhline(0, color=C_BLACK, linewidth=0.6)
fig.tight_layout()
plt.show()
print("Higher hold more than offset weaker handle in dollars. Relative to the market, lost handle share remains the cautionary signal.")

# 5. Competitor handle-share bridge
bridge_plot = bridge.sort_values("share_pp_change")
fig, ax = plt.subplots(figsize=(8.2, 4.6))
ypos = range(len(bridge_plot))
bar_colors = [C_ORANGE if v > 0 else C_BLUE for v in bridge_plot["share_pp_change"]]
ax.barh(list(ypos), bridge_plot["share_pp_change"], color=bar_colors, zorder=2)
ax.axvline(0, color=C_BLACK, linewidth=0.9, zorder=3)
ax.set_yticks(list(ypos), bridge_plot["operator"])
ax.set_xlabel("Handle-share change (pp), Jan–Jul 2026 vs 2025")
ax.set_ylabel("Native reported operator name")
ax.set_title("Which operators gained or lost handle share?")
xmin = float(bridge_plot["share_pp_change"].min()) - 1.0
xmax = float(bridge_plot["share_pp_change"].max()) + 0.8
ax.set_xlim(xmin, xmax)
for i, (_, r) in enumerate(bridge_plot.iterrows()):
    ax.text(
        r["share_pp_change"] + (0.08 if r["share_pp_change"] >= 0 else -0.08),
        i,
        f"{r['share_pp_change']:+.2f}",
        va="center",
        ha="left" if r["share_pp_change"] >= 0 else "right",
        fontsize=8,
    )
ax.text(0.99, 0.02, "ESPN Bet and theScore Bet kept separate;\ncontinuity is not assumed.",
        transform=ax.transAxes, ha="right", va="bottom", fontsize=8, color=C_GRAY)
fig.tight_layout()
plt.show()
print("Fanatics is the largest clearly comparable share gainer. ESPN Bet / theScore Bet are native-name transitions, not a mapped entity.")

# 6. Hold and deduction context
hold_2025 = monthly[monthly["period_start"].isin(ytd_2025)].sort_values("period_start")
hold_2026 = monthly[monthly["period_start"].isin(ytd_2026)].sort_values("period_start")
rel_hold = (100 * (hold_2026["fanduel_hold"].to_numpy() - hold_2026["market_hold"].to_numpy()))
# matched-month: FanDuel hold minus statewide hold in the SAME year-month; user asked by matched month
# Show 2026 minus market 2026 and also 2025 as separate series without crossing years.
rel_hold_2025 = 100 * (hold_2025["fanduel_hold"].to_numpy() - hold_2025["market_hold"].to_numpy())
rel_hold_2026 = 100 * (hold_2026["fanduel_hold"].to_numpy() - hold_2026["market_hold"].to_numpy())
fig, axes = plt.subplots(1, 2, figsize=(9.6, 4.1))
ax = axes[0]
ax.axhline(0, color=C_BLACK, linewidth=0.9)
ax.plot(months_abbr, rel_hold_2025, marker="o", color=C_GRAY, linewidth=2)
ax.plot(months_abbr, rel_hold_2026, marker="o", color=C_ORANGE, linewidth=2)
ax.text(0, rel_hold_2025[0], "2025", color=C_GRAY, fontsize=8)
ax.text(0, rel_hold_2026[0], "2026", color=C_ORANGE, fontsize=8)
ax.set_title("A. Hold advantage vs statewide")
ax.set_xlabel("Matched month (Jan–Jul)")
ax.set_ylabel("FanDuel hold − statewide hold (pp)")
ax = axes[1]
ded_jj = ded[ded["period"].eq("Jan-Jul")]
cats = ["FanDuel", "Statewide"]
x = range(len(cats))
w = 0.35
v25 = [100 * float(ded_jj[(ded_jj.entity == e) & (ded_jj.year_side == "2025")]["reported_deduction_intensity"].iloc[0]) for e in cats]
v26 = [100 * float(ded_jj[(ded_jj.entity == e) & (ded_jj.year_side == "2026")]["reported_deduction_intensity"].iloc[0]) for e in cats]
ax.bar([i - w / 2 for i in x], v25, width=w, color=C_GRAY, label="2025")
ax.bar([i + w / 2 for i in x], v26, width=w, color=C_SKY, label="2026")
ax.set_xticks(list(x), cats)
ax.set_ylabel("Reported deduction intensity (%)")
ax.set_xlabel("Jan–Jul Accrual Win − taxable, as % of Accrual Win")
ax.set_title("B. Promotions and other reported adjustments")
ax.legend(title="Year", loc="upper right", frameon=False)
for i, (a, b) in enumerate(zip(v25, v26)):
    ax.text(i - w / 2, a + 0.03, f"{a:.2f}", ha="center", fontsize=8)
    ax.text(i + w / 2, b + 0.03, f"{b:.2f}", ha="center", fontsize=8)
fig.suptitle("Did FanDuel’s MA hold advantage persist, and did reported deductions rise?", y=1.02)
fig.tight_layout()
plt.show()
print("Hold advantage stayed positive. Reported deduction intensity is promotions and other reported adjustments, not pure promotional spend.")
print("VISUAL_CONCLUSION: YTD cautionary; direction improving; hold offset weaker handle; Fanatics largest comparable gainer; July unproven; MA is not a nationwide FLUT proxy.")


## 9. Source evidence (complete provenance for 14 months)


In [ ]:
if not ALL_GATES_PASSED:
    raise SystemExit("Eligibility gates failed.")

src = official[[
    "period_start", "report_status", "source_file", "source_url", "source_sha256",
    "handle", "gross_revenue",
]].sort_values("period_start").copy()
fd_src = fanduel[["period_start", "source_sha256"]].rename(columns={"source_sha256": "fanduel_source_sha256"})
src = src.merge(fd_src, on="period_start", validate="one_to_one")
src["same_hash_as_fanduel"] = src["source_sha256"].eq(src["fanduel_source_sha256"])
assert src["same_hash_as_fanduel"].all()

print("=== Complete source provenance ===")
for _, r in src.iterrows():
    print("-" * 80)
    print(f"period:        {r['period_start'].date()}")
    print(f"report_status: {r['report_status']}")
    print(f"source_file:   {r['source_file']}")
    print(f"source_url:    {r['source_url']}")
    print(f"source_sha256: {r['source_sha256']}")
    print(f"same_hash_as_fanduel: {r['same_hash_as_fanduel']}")

display(src[[
    "period_start", "report_status", "source_file", "source_url", "source_sha256", "same_hash_as_fanduel"
]])


## 10. Limitations

- Massachusetts online sports betting only; not a national FLUT proxy.
- Accrual Win is distinct from Taxable Gaming Revenue.
- Metric-note definition validity dates remain unknown; gate 0 is a scoped window approval only.
- Exact operator name `FanDuel` only; Fanatics is separate.
- `ESPN Bet` / `theScore Bet` / `theScore Bet*` are not silently mapped.
- Reported deduction gap is not pure promotions and not an EBITDA expense.
- July is one month of Q3 evidence and is not a confirmed reversal.
- No HHI, forecasts, consensus, valuation, or trading recommendation.


## 11. Level vs direction — what the evidence says


In [ ]:
if not ALL_GATES_PASSED:
    raise SystemExit("Eligibility gates failed.")

q1 = agg.loc[agg["key"].eq("Q1")].iloc[0]
q2 = agg.loc[agg["key"].eq("Q2")].iloc[0]
july = agg.loc[agg["key"].eq("July")].iloc[0]
jj = jan_jul

assert abs(q1["fanduel_handle_share_pp"] - (-3.06)) < 0.05
assert abs(q2["fanduel_handle_share_pp"] - (-1.57)) < 0.05
assert abs(july["fanduel_handle_share_pp"] - 0.49) < 0.05
assert abs(jj["fanduel_handle_share_pp"] - (-2.04)) < 0.05

ytd_position = "cautionary"
direction = "improving"
durability = "unproven"

fd_jj = decomp[(decomp.period == "Jan-Jul") & (decomp.entity == "FanDuel")].iloc[0]

facts = f"""
Observed facts:
- Jan-Jul market handle YoY {jj['market_handle_yoy_pct']:+.2f}%; market Accrual Win YoY {jj['market_accrual_win_yoy_pct']:+.2f}%.
- Jan-Jul FanDuel handle YoY {jj['fanduel_handle_yoy_pct']:+.2f}%; FanDuel Accrual Win YoY {jj['fanduel_accrual_win_yoy_pct']:+.2f}%.
- FanDuel handle share {100*jj['prior_fanduel_handle_share']:.2f}% -> {100*jj['current_fanduel_handle_share']:.2f}% ({jj['fanduel_handle_share_pp']:+.2f} pp).
- Handle-share YoY by segment: Q1 {q1['fanduel_handle_share_pp']:+.2f} pp; Q2 {q2['fanduel_handle_share_pp']:+.2f} pp; July {july['fanduel_handle_share_pp']:+.2f} pp.
- Jan-Jul FanDuel revenue change decomposition: handle_effect {money(fd_jj['handle_effect_$'])} ({fd_jj['handle_effect_pct_of_prior_revenue']:+.2f}% of prior revenue);
  hold_effect {money(fd_jj['hold_effect_$'])} ({fd_jj['hold_effect_pct_of_prior_revenue']:+.2f}% of prior revenue).
"""

interp = f"""
Level vs direction (MA only):
1. YTD relative position: {ytd_position}.
   FanDuel underperformed the MA market on handle and Accrual Win for Jan-Jul, and lost handle share ({jj['fanduel_handle_share_pp']:+.2f} pp).
2. Recent direction: {direction}.
   Year-over-year handle-share pressure eased from Q1 ({q1['fanduel_handle_share_pp']:+.2f} pp) to Q2 ({q2['fanduel_handle_share_pp']:+.2f} pp) to July ({july['fanduel_handle_share_pp']:+.2f} pp).
3. Durability / confidence: {durability}.
   July is a single month of partial Q3 evidence. It is not a confirmed reversal.

Driver (exact decomposition, Jan-Jul FanDuel):
- Accrual Win still rose because a positive hold effect outweighed a negative handle effect.
- Relative to the market, share loss remains the dominant cautionary signal.
"""

uncertainty = """
Unresolved uncertainty:
- Definition validity outside the scoped approved window.
- Whether native FanDuel naming covers all FLUT-relevant MA sportsbook activity.
- Corporate continuity between ESPN Bet and theScore Bet labels (not assumed).
- Translation from MA sportsbook share/hold to FLUT consolidated results (not estimated).
"""

print(facts)
print(interp)
print(uncertainty)
print(f"YTD_RELATIVE_POSITION={ytd_position}")
print(f"RECENT_DIRECTION={direction}")
print(f"DURABILITY_CONFIDENCE={durability}")


In [ ]:
hash_original_after = sha256(ORIGINAL_DB)
hash_staging_after = sha256(STAGING_DB)
assert hash_original_after == hash_original_before, "original database hash changed"
assert hash_staging_after == hash_staging_before, "staging database hash changed"
print("Database hashes unchanged.")
print(f"approved_pipeline_commit={approved_pipeline_commit}")
print(f"current_analysis_commit={current_analysis_commit}")
print(f"staging_sha256={hash_staging_after}")
print(f"original_sha256={hash_original_after}")
